# Download data from Google Earth Engine
This code snippet provides an example of accessing and downloading data from Google Earth Engine using the earthengine-api for Python (read the [docs](https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api)).

You will first need to register for a (free) Google Cloud project account.

In this snippet, we will download data from Sentinel-2. You can browse the available datasets in the [Earth Engine Catalog](https://developers.google.com/earth-engine/datasets/catalog). There is also the [awesome-gee-community-catalog](https://gee-community-catalog.org/projects/) providing additional datasets contributed by the open-source community.

## Setup

In [ ]:
# Install earthengine-api
# !pip install earthengine-api
# !pip install geemap
!pip install geedim
!pip install --upgrade --pre xee

In [ ]:
from pathlib import Path
import ee
import geemap
import xee
from xee import helpers
import xarray as xr
from shapely import box
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Trigger the authentication flow
ee.Authenticate()

# Initialize the library
# You will need to follow the link, then copy and paste the verification code
ee.Initialize() #project='my-project'

## Data search

Here is where you'll **update using your own search parameters:**
* The sensor / product name: known in Google Earth Engine as the **image collection** (`ee.ImageCollection`)
* The region: here defined using a **bounding box**
* The time period: here defined using a **start and end date**
* Count: max number of images to return.

Visit the data catalog to determine the bands you want. Example for Sentinel-2: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED#bands

In [ ]:
# Define collection
collection_name = 'COPERNICUS/S2_SR_HARMONIZED'

# Define bands to select fromm the collection
bands = ['B1','B2','B3','B4','B5','B6','B7','B8A','B11','B12']

# Define start and end dates for search
start_date, end_date = '2019-05-01', '2019-05-10'  

# Define max number of images to return
count = 100

# Define region of interest (roi) using bounding box geometry
region = box(-97.7, 42.8, -97.5, 43.0)

In [ ]:
# Convert the region to ee feature collection
region_gdf = gpd.GeoDataFrame(geometry=[region], crs=4326)
region_geometry = geemap.geopandas_to_ee(region_gdf).geometry()

print(f"Searching collection {collection_name} between {start_date} and {end_date} for region {region.bounds}.")

# Filter the image collection for the region and dates
collection = ee.ImageCollection(collection_name)\
                .filterDate(start_date, end_date)\
                .filterBounds(region_geometry)\
                .limit(count)
               #.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20))\ # optionally filter by cloud pixel pct band for S2
                
# Print the number of images retured
collection_size = collection.size().getInfo()
print(f"Search returned {collection_size} images:")

# Get image IDs of images in the collection
image_ids = [f'{collection_name}/{im_id}' for im_id in collection.aggregate_array('system:index').getInfo()]
print(image_ids)

# Show the collection
collection

## Open or download data

You have the option to either stream the data into Python or download the data. 

In both cases, you will first need to specify the files from your search that you want to download. You can either download all of them, or choose specific files from the list we just got.

### Data streaming

To stream our filtered collection from Google Earth Engine directly into Python, we'll use the [xee library](https://github.com/google/Xee).

In [ ]:
grid = helpers.extract_grid_params(collection)
print(grid)

# Load the collection into a xr dataset
dataset = xr.open_dataset(collection.select(bands), engine='ee', **grid)

# Convert the datset with bands as separate variables into a single data array with band dimension
# This step may take some time to run
data = dataset.to_array(dim="band").rio.write_crs(grid['crs'])
print(f'Loaded data with shape: {data.shape}')
data

In [ ]:
# Quick basic RGB plot for the first image in the image collection

# Get RGB data cube, linear stretch normalized
rgb = data.sel(band=['B4', 'B3', 'B2'])
rgb = (rgb / 3000).clip(0, 1)

# RGB plot
rgb.isel(time=0).plot.imshow()
region_gdf.to_crs(rgb.rio.crs).plot(ax=plt.gca(), facecolor='none', edgecolor='r')

In [ ]:
# SWIR band plot all timestamps
swir = data.sel(band='B11')
swir.plot(col='time', vmin=0, vmax=5000)

### Data download

There are a few different ways to download data from GEE. You could for example use the xr.DataArray we loaded using xee and write that out to GeoTIFF or NetCDF. A lot of researchers use geemap methods to download data directly from GEE: https://book.geemap.org/chapters/07_data_export.html

In [ ]:
# Get an image for the first image in our returned collection, select RGB bands for visualizing
image_to_viz = ee.Image(image_ids[0]).select(['B4', 'B3', 'B2'])

# Lets do a qucik interative map through geemap
Map = geemap.Map()

vis_params = {'min': 0, 'max': 3000, 'gamma': [0.95, 1.1, 1]}

Map.centerObject(image_to_viz, 9)
Map.addLayer(image_to_viz, vis_params, 'Sentinel-2')
Map.addLayer(region_geometry, {'color': 'FF0000'}, ' Region')
Map

### Download

In [ ]:
# Define path to download to
downloadPath = Path('data/downloads/GEE_Sentinel2') # MODIFY
downloadPath.mkdir(exist_ok=True, parents=True) # Create download directory if does not already exist

In [ ]:
# Download using geemap

# Download the first image in our collection for the bands we specified earlier
image_to_download = ee.Image(image_ids[0]).select(bands)
image_name = image_to_download.get('system:index').getInfo()
image_crs, image_scale = grid['crs'], grid['crs_transform'][0]
print(f'Downloading {image_name} in CRS {image_crs} at scale {image_scale}')

geemap.download_ee_image(image_to_download, str(downloadPath/f'{image_name}.tif'), 
                         scale=image_scale, crs=image_crs, region=region_geometry)

In [ ]:
# Download using geemap fishnet to download in smaller tiles
# Use this option if trying to download a large size area with many pixels
gridFeatures = geemap.fishnet(geom, rows=nrows, cols=ncols)
geemap.download_ee_image_tiles(image_to_download, gridFeatures, 
                               str(downloadPath), prefix=f'{image_name}_', 
                               scale=image_scale, crs=image_crs)